# App Probe Check

Deterministic smoke task. Proves, from a job, that the probe identity can make authenticated requests to the manifest's apps and that the job can read their trace sources, which the runtime gate needs for `probes`, `min_traces` and `max_trace_errors`.

**Probe identity.** A service principal (OAuth M2M) whose client id and secret come from a Databricks secret scope: `probe_auth: {secret_scope, client_id_key, client_secret_key}` on the target, or top-level `probe_auth` for all app targets. It is required for probing: Apps accept only OAuth tokens, and a job's own credential is redirected to sign-in, even with `CAN_MANAGE` on the app (verified from a job run). The service principal needs `CAN_USE` on the app; the job's run-as identity needs `READ` on the secret scope.

For every `app` runtime target with `probes` or `probe_auth`, it runs the target's `probes` (default `GET /`) against the URL from `w.apps.get(name)`, and reads `telemetry.traces_table`, `telemetry.logs_table` and `telemetry.mlflow_experiment_id` when set. Redirects are not followed; a redirect to sign-in means the credential was not accepted.

For every `genie_space` runtime target it checks that the job's identity can read the space's configuration (`get_space(include_serialized_space=True)`, which needs `CAN_EDIT`); the gate needs that for its table and example SQL checks. A space that does not exist yet is `skipped`.

| Result | Meaning |
|---|---|
| `pass` | Authenticated request accepted, probe expectations met |
| `fail` | Bad SPN credentials, auth rejected (redirect, 401, 403), a probe expectation failed, a trace source is unreadable, or a Genie space configuration is unreadable (no `CAN_EDIT`); the task fails |
| `skipped` | App does not exist yet or is not `RUNNING` and its trace sources are readable (nothing to probe) |

In [0]:
%pip install --upgrade databricks-sdk mlflow
dbutils.library.restartPython()

In [0]:
import json
import os

dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")
dbutils.widgets.text("build_root", "", "Build root (resolves a relative manifest_path)")

manifest_path = dbutils.widgets.get("manifest_path").strip()
build_root = dbutils.widgets.get("build_root").rstrip("/")
if manifest_path and not os.path.isabs(manifest_path):
    manifest_path = os.path.join(build_root, manifest_path)
if not manifest_path:
    raise ValueError("manifest_path is required")

with open(manifest_path) as f:
    manifest = json.load(f)

targets = manifest.get("runtime_targets") or {}
app_targets = {n: t for n, t in targets.items() if t.get("type") == "app"}
genie_targets = {n: t for n, t in targets.items() if t.get("type") == "genie_space"}
print(f"manifest: {manifest_path}")
print(f"app runtime targets: {sorted(app_targets) or 'none'}")
print(f"genie_space runtime targets: {sorted(genie_targets) or 'none'}")

In [0]:
import sys

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound

# app_probes.py sits next to this notebook (cwd for workspace notebooks).
sys.path.insert(0, os.getcwd())
from datetime import datetime, timedelta, timezone

from app_probes import count_mlflow_traces, probe_client, run_app_probes

w = WorkspaceClient()
identity = w.current_user.me().user_name
print(f"job identity: {identity}")


def check_trace_sources(tgt):
    """Can this job read the target's trace sources? (OTel spans table, MLflow 3 experiment)"""
    tel, out = tgt.get("telemetry") or {}, {}
    since = datetime.now(timezone.utc) - timedelta(days=1)
    for key in ("traces_table", "logs_table"):
        if tel.get(key):
            try:
                n = spark.sql(f"SELECT count(*) AS n FROM {tel[key]} WHERE date >= date(:d)",
                              args={"d": since.date().isoformat()}).collect()[0]["n"]
                out[key] = {"result": "pass", "rows_last_day": n}
            except Exception as e:
                out[key] = {"result": "fail", "detail": f"{type(e).__name__}: {str(e)[:200]}"}
    if tel.get("mlflow_experiment_id"):
        try:
            r = count_mlflow_traces(tel["mlflow_experiment_id"], int(since.timestamp() * 1000), cap=100)
            out["mlflow_experiment_id"] = {"result": "pass", "traces_last_day": r["traces"]}
        except Exception as e:
            out["mlflow_experiment_id"] = {"result": "fail", "detail": f"{type(e).__name__}: {str(e)[:200]}"}
    return out


results = {}
for name, tgt in app_targets.items():
    app_name = tgt["name"]
    sources = check_trace_sources(tgt)
    src_probs = [f"{k}: {v['detail']}" for k, v in sources.items() if v["result"] == "fail"]
    auth = tgt.get("probe_auth") or manifest.get("probe_auth")
    if not tgt.get("probes") and not auth:
        # Nothing for the gate to probe; only the trace sources matter.
        results[name] = {"app": app_name, "result": "fail" if src_probs else "pass", "trace_sources": sources,
                         "problems": src_probs, "detail": "no probes or probe_auth: not probed"}
        continue
    try:
        app = w.apps.get(app_name)
    except Exception as e:
        results[name] = {"app": app_name, "result": "fail" if src_probs else "skipped", "trace_sources": sources,
                         "problems": src_probs, "detail": f"app not found ({type(e).__name__})"}
        continue
    state = getattr(getattr(app.app_status, "state", None), "value", None)
    compute = getattr(getattr(app.compute_status, "state", None), "value", None)
    if state != "RUNNING" or compute != "ACTIVE":
        results[name] = {"app": app_name, "result": "fail" if src_probs else "skipped", "trace_sources": sources,
                         "problems": src_probs, "detail": f"app_status={state} compute={compute}"}
        continue
    try:
        pw, probe_identity = probe_client(w, auth, dbutils.secrets.get)
        probe_results, probs = run_app_probes(w, app_name, tgt.get("probes"), probe_w=pw)
    except Exception as e:  # no probe_auth, or missing secret scope/key
        probe_identity, probe_results, probs = "unresolved", [], [f"probe_auth: {type(e).__name__}: {str(e)[:300]}"]
    probs += src_probs
    results[name] = {"app": app_name, "result": "fail" if probs else "pass", "probe_identity": probe_identity,
                     "probes": probe_results, "trace_sources": sources, "problems": probs}

def check_genie_access(tgt):
    """The gate reads the space's configuration, which needs CAN_EDIT for this job's identity."""
    label = tgt.get("id") or tgt.get("name")
    sid = tgt.get("id")
    if not sid:
        matches, token = [], None
        while True:
            r = w.genie.list_spaces(page_size=100, page_token=token)
            matches += [s.space_id for s in (r.spaces or []) if s.title == tgt.get("name")]
            token = r.next_page_token
            if not token:
                break
        if not matches:
            return {"space": label, "result": "skipped", "detail": "no space with this title yet"}
        if len(matches) > 1:
            return {"space": label, "result": "fail", "problems": [f"title matches {len(matches)} spaces; set id"]}
        sid = matches[0]
    try:
        sp = w.genie.get_space(sid, include_serialized_space=True)
    except NotFound:
        return {"space": label, "result": "skipped", "detail": f"space {sid} does not exist yet"}
    except Exception as e:  # PermissionDenied: no CAN_EDIT
        return {"space": label, "result": "fail",
                "problems": [f"cannot read configuration as {identity} ({type(e).__name__}); grant CAN_EDIT"]}
    if not sp.serialized_space:
        return {"space": label, "result": "fail", "problems": [f"configuration not returned to {identity}; grant CAN_EDIT"]}
    return {"space": label, "space_id": sid, "result": "pass", "warehouse_id": sp.warehouse_id}


for name, tgt in genie_targets.items():
    results[name] = check_genie_access(tgt)

for name, r in results.items():
    print(f"  {name} ({r.get('app') or r.get('space')}) as {r.get('probe_identity', '-')}: {r['result'].upper()} "
          f"{r.get('problems') or r.get('detail') or ''}")

summary = {
    "identity": identity,
    "status": "skipped" if not results or all(r["result"] == "skipped" for r in results.values())
    else ("fail" if any(r["result"] == "fail" for r in results.values()) else "pass"),
    "targets": results,
}
print(f"app_probe {summary['status']}")
if summary["status"] == "fail":
    # Fail the task so the setup notebook's smoke summary shows it.
    raise RuntimeError(f"app probes / Genie space access failed (job identity {identity}): " + json.dumps(
        {n: {"as": r.get("probe_identity", identity), "problems": r.get("problems")}
         for n, r in results.items() if r["result"] == "fail"}))
dbutils.notebook.exit(json.dumps(summary, default=str))